# Model Training

In [286]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [287]:
df = pd.read_csv('C:/Users/mrcoo/ml_course/ML-course/lesson_11/Indian_Kids_Screen_Time.csv')

In [288]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Gender                             9712 non-null   object 
 2   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 3   Primary_Device                     9712 non-null   object 
 4   Exceeded_Recommended_Limit         9712 non-null   bool   
 5   Educational_to_Recreational_Ratio  9712 non-null   float64
 6   Health_Impacts                     6494 non-null   object 
 7   Urban_or_Rural                     9712 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(4)
memory usage: 540.7+ KB


In [289]:
df.sample(5)

,Age,Gender,Avg_Daily_Screen_Time_hr,Primary_Device,Exceeded_Recommended_Limit,Educational_to_Recreational_Ratio,Health_Impacts,Urban_or_Rural
4079,18,Female,6.97,Smartphone,True,0.38,Poor Sleep,Urban
4773,13,Female,5.44,Smartphone,True,0.32,"Poor Sleep, Obesity Risk",Urban
7529,14,Male,4.26,Smartphone,True,0.47,Poor Sleep,Rural
8424,18,Female,2.49,TV,False,0.44,NaN,Rural
8090,13,Male,5.05,Smartphone,True,0.44,"Poor Sleep, Anxiety",Rural


In [290]:
df.isna().sum()

Age                                     0
Gender                                  0
Avg_Daily_Screen_Time_hr                0
Primary_Device                          0
Exceeded_Recommended_Limit              0
Educational_to_Recreational_Ratio       0
Health_Impacts                       3218
Urban_or_Rural                          0
dtype: int64

In [291]:
df['Health_Impacts'].value_counts()

Health_Impacts
Poor Sleep                                       2268
Poor Sleep, Eye Strain                            979
Eye Strain                                        644
Poor Sleep, Anxiety                               608
Poor Sleep, Obesity Risk                          452
Anxiety                                           385
Poor Sleep, Eye Strain, Anxiety                   258
Obesity Risk                                      252
Poor Sleep, Eye Strain, Obesity Risk              188
Eye Strain, Anxiety                               135
Eye Strain, Obesity Risk                          106
Poor Sleep, Anxiety, Obesity Risk                  78
Anxiety, Obesity Risk                              69
Poor Sleep, Eye Strain, Anxiety, Obesity Risk      37
Eye Strain, Anxiety, Obesity Risk                  35
Name: count, dtype: int64

In [292]:
df['Health_Impacts'].fillna(df['Health_Impacts'].mode()[0], inplace=True)

In [293]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Gender                             9712 non-null   object 
 2   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 3   Primary_Device                     9712 non-null   object 
 4   Exceeded_Recommended_Limit         9712 non-null   bool   
 5   Educational_to_Recreational_Ratio  9712 non-null   float64
 6   Health_Impacts                     9712 non-null   object 
 7   Urban_or_Rural                     9712 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(4)
memory usage: 540.7+ KB


### Converting boolean datatype into int

In [294]:
df['Exceeded_Recommended_Limit'] = df['Exceeded_Recommended_Limit'].astype(int)

In [295]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Gender                             9712 non-null   object 
 2   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 3   Primary_Device                     9712 non-null   object 
 4   Exceeded_Recommended_Limit         9712 non-null   int32  
 5   Educational_to_Recreational_Ratio  9712 non-null   float64
 6   Health_Impacts                     9712 non-null   object 
 7   Urban_or_Rural                     9712 non-null   object 
dtypes: float64(2), int32(1), int64(1), object(4)
memory usage: 569.2+ KB


| **Encoding Type**      | **When to Use**                                                              | **Code Example**                                                                            | **Pros**                                   | **Cons**                                                   |
| ---------------------- | ---------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------- | ------------------------------------------ | ---------------------------------------------------------- |
| **One-Hot Encoding**   | Categorical data with **low cardinality** (few unique values)                | `pd.get_dummies(df, columns=['Col'], prefix='col', dtype=int)`                              | Simple, no ordinal assumption              | High dimensionality with many unique values                |
| **Label Encoding**     | Categorical data with **natural order** or for tree-based models             | `df['Col'] = encoder.fit_transform(df['Col'])`                                              | Efficient, preserves order (if meaningful) | May imply ordinal relationship that doesn't exist          |
| **Frequency Encoding** | Categorical features where frequency may be meaningful                       | `df['Col_freq'] = df['Col'].map(df['Col'].value_counts())`                                  | Simple, low dimensional                    | Doesn't capture relationship with target                   |
| **Target Encoding**    | When there's strong correlation between category and target (numeric target) | `df['Col_encoded'] = df['Col'].map(df.groupby('Col')['target'].mean())`                     | Powerful, captures target relationship     | Risk of **data leakage**; must be cross-validated properly |
| **Ordinal Encoding**   | Ordered categories (e.g., education level, ratings)                          | `df['Col_ordinal'] = df['Col'].map(custom_mapping_dict)`                                    | Preserves order, compact                   | Requires manual mapping                                    |
| **Binary Encoding**    | High cardinality categorical features                                        | `import category_encoders as ce`<br>`df = ce.BinaryEncoder(cols=['Col']).fit_transform(df)` | Lower dimensionality than one-hot          | Slightly harder to interpret                               |
| **Hash Encoding**      | High-cardinality categorical features (e.g., text categories)                | `ce.HashingEncoder(cols=['Col'])`                                                           | Fast and memory-efficient                  | Possibility of collisions, not interpretable               |
| **Boolean to Int**     | Boolean columns                                                              | `df['Col'] = df['Col'].astype(int)`                                                         | Easy, direct                               | Only for True/False values                                 |
| **Custom Mapping**     | Any manual logic                                                             | `df['Col'] = df['Col'].map({'A': 0, 'B': 1})`                                               | Full control                               | Prone to human error                                       |


### ENCODING GENDER COLUMN WITH ONE-HOT ENCODING

In [296]:
df = pd.get_dummies(df, columns=['Gender'], prefix='gender', dtype=int)

In [297]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 9 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 2   Primary_Device                     9712 non-null   object 
 3   Exceeded_Recommended_Limit         9712 non-null   int32  
 4   Educational_to_Recreational_Ratio  9712 non-null   float64
 5   Health_Impacts                     9712 non-null   object 
 6   Urban_or_Rural                     9712 non-null   object 
 7   gender_Female                      9712 non-null   int32  
 8   gender_Male                        9712 non-null   int32  
dtypes: float64(2), int32(3), int64(1), object(3)
memory usage: 569.2+ KB


In [298]:
df['Primary_Device'].value_counts()

Primary_Device
Smartphone    4568
TV            2487
Laptop        1433
Tablet        1224
Name: count, dtype: int64

### PRIMARY_DEVICE COLUMN ENCODED

In [299]:
df['Primary_Device_freq'] = df['Primary_Device'].map(df['Primary_Device'].value_counts())

In [300]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 2   Primary_Device                     9712 non-null   object 
 3   Exceeded_Recommended_Limit         9712 non-null   int32  
 4   Educational_to_Recreational_Ratio  9712 non-null   float64
 5   Health_Impacts                     9712 non-null   object 
 6   Urban_or_Rural                     9712 non-null   object 
 7   gender_Female                      9712 non-null   int32  
 8   gender_Male                        9712 non-null   int32  
 9   Primary_Device_freq                9712 non-null   int64  
dtypes: float64(2), int32(3), int64(2), object(3)
memory usage: 645.1+ KB


In [301]:
df.drop(columns='Primary_Device', axis=1, inplace=True)

In [302]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 9 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 2   Exceeded_Recommended_Limit         9712 non-null   int32  
 3   Educational_to_Recreational_Ratio  9712 non-null   float64
 4   Health_Impacts                     9712 non-null   object 
 5   Urban_or_Rural                     9712 non-null   object 
 6   gender_Female                      9712 non-null   int32  
 7   gender_Male                        9712 non-null   int32  
 8   Primary_Device_freq                9712 non-null   int64  
dtypes: float64(2), int32(3), int64(2), object(2)
memory usage: 569.2+ KB


In [303]:
df['Health_Impacts'] = df['Health_Impacts'].map(df['Health_Impacts'].value_counts())

In [304]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 9 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 2   Exceeded_Recommended_Limit         9712 non-null   int32  
 3   Educational_to_Recreational_Ratio  9712 non-null   float64
 4   Health_Impacts                     9712 non-null   int64  
 5   Urban_or_Rural                     9712 non-null   object 
 6   gender_Female                      9712 non-null   int32  
 7   gender_Male                        9712 non-null   int32  
 8   Primary_Device_freq                9712 non-null   int64  
dtypes: float64(2), int32(3), int64(3), object(1)
memory usage: 569.2+ KB


In [305]:
df['Urban_or_Rural'].value_counts()

Urban_or_Rural
Urban    6851
Rural    2861
Name: count, dtype: int64

In [306]:
df = pd.get_dummies(df, columns=['Urban_or_Rural'], prefix='URB_RUR', dtype=int)

In [307]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9712 entries, 0 to 9711
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Age                                9712 non-null   int64  
 1   Avg_Daily_Screen_Time_hr           9712 non-null   float64
 2   Exceeded_Recommended_Limit         9712 non-null   int32  
 3   Educational_to_Recreational_Ratio  9712 non-null   float64
 4   Health_Impacts                     9712 non-null   int64  
 5   gender_Female                      9712 non-null   int32  
 6   gender_Male                        9712 non-null   int32  
 7   Primary_Device_freq                9712 non-null   int64  
 8   URB_RUR_Rural                      9712 non-null   int32  
 9   URB_RUR_Urban                      9712 non-null   int32  
dtypes: float64(2), int32(5), int64(3)
memory usage: 569.2 KB


In [308]:
df.sample(15)

,Age,Avg_Daily_Screen_Time_hr,Exceeded_Recommended_Limit,Educational_to_Recreational_Ratio,Health_Impacts,gender_Female,gender_Male,Primary_Device_freq,URB_RUR_Rural,URB_RUR_Urban
597,16,3.74,1,0.35,608,1,0,1224,1,0
5564,10,5.56,1,0.47,5486,1,0,2487,0,1
8007,17,5.57,1,0.33,5486,0,1,4568,1,0
5751,16,3.23,1,0.34,644,1,0,4568,0,1
2629,17,6.56,1,0.48,5486,1,0,4568,1,0
9423,12,1.94,0,0.37,5486,1,0,1433,1,0
1133,18,2.52,0,0.38,5486,0,1,4568,1,0
1324,10,3.05,1,0.58,452,1,0,4568,0,1
7580,8,7.83,1,0.44,5486,0,1,2487,0,1
8662,16,6.96,1,0.47,452,0,1,2487,0,1


In [309]:
num_col=df.select_dtypes(include=['int64','float64','int32']).columns.tolist()

In [310]:
num_col

['Age',
 'Avg_Daily_Screen_Time_hr',
 'Exceeded_Recommended_Limit',
 'Educational_to_Recreational_Ratio',
 'Health_Impacts',
 'gender_Female',
 'gender_Male',
 'Primary_Device_freq',
 'URB_RUR_Rural',
 'URB_RUR_Urban']

In [311]:
minmaxscal = MinMaxScaler()

In [312]:
df[num_col] = minmaxscal.fit_transform(df[num_col])

In [313]:
df

,Age,Avg_Daily_Screen_Time_hr,Exceeded_Recommended_Limit,Educational_to_Recreational_Ratio,Health_Impacts,gender_Female,gender_Male,Primary_Device_freq,URB_RUR_Rural,URB_RUR_Urban
0,0.6,0.287257,1.0,0.400000,0.173179,0.0,1.0,1.000000,0.0,1.0
1,0.3,0.331893,1.0,0.000000,1.000000,1.0,0.0,0.062500,0.0,1.0
2,1.0,0.268539,1.0,0.066667,1.000000,1.0,0.0,0.377691,0.0,1.0
3,0.7,0.087113,0.0,0.300000,1.000000,1.0,0.0,0.062500,0.0,1.0
4,0.4,0.424046,1.0,0.633333,0.105118,1.0,0.0,1.000000,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
9707,0.9,0.234701,1.0,0.466667,1.000000,0.0,1.0,1.000000,0.0,1.0
9708,0.9,0.318934,1.0,0.333333,1.000000,1.0,0.0,1.000000,1.0,0.0
9709,0.8,0.404608,1.0,0.300000,0.040910,0.0,1.0,1.000000,1.0,0.0
9710,0.9,0.403168,1.0,0.433333,1.000000,0.0,1.0,0.377691,0.0,1.0


# MODEL TRAINING


### input

In [ ]:
x = df.drop('URB_RUR_Urban', axis=1)

In [315]:
x

,Age,Avg_Daily_Screen_Time_hr,Exceeded_Recommended_Limit,Educational_to_Recreational_Ratio,gender_Female,gender_Male,Primary_Device_freq,URB_RUR_Rural,URB_RUR_Urban
0,0.6,0.287257,1.0,0.400000,0.0,1.0,1.000000,0.0,1.0
1,0.3,0.331893,1.0,0.000000,1.0,0.0,0.062500,0.0,1.0
2,1.0,0.268539,1.0,0.066667,1.0,0.0,0.377691,0.0,1.0
3,0.7,0.087113,0.0,0.300000,1.0,0.0,0.062500,0.0,1.0
4,0.4,0.424046,1.0,0.633333,1.0,0.0,1.000000,0.0,1.0
...,...,...,...,...,...,...,...,...,...
9707,0.9,0.234701,1.0,0.466667,0.0,1.0,1.000000,0.0,1.0
9708,0.9,0.318934,1.0,0.333333,1.0,0.0,1.000000,1.0,0.0
9709,0.8,0.404608,1.0,0.300000,0.0,1.0,1.000000,1.0,0.0
9710,0.9,0.403168,1.0,0.433333,0.0,1.0,0.377691,0.0,1.0


In [332]:
y = df['URB_RUR_Urban'].astype(int)

In [333]:
y

0       1
1       1
2       1
3       1
4       1
       ..
9707    1
9708    0
9709    0
9710    1
9711    1
Name: URB_RUR_Urban, Length: 9712, dtype: int32

splitting data into train and test

In [334]:
from sklearn.ensemble import RandomForestRegressor



In [335]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

In [337]:
model=DecisionTreeClassifier()

In [338]:
model

DecisionTreeClassifier()

In [339]:
model.fit(x_train, y_train)


DecisionTreeClassifier()

### Prediction

In [340]:
y_predict = model.predict(x_test)

In [341]:
y_predict

array([1, 1, 1, ..., 0, 1, 0])

In [342]:
acc = accuracy_score(y_test, y_predict)

In [343]:
acc

1.0